In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


#df = pd.read_pickle(r"S:\Sachuriga\Ephys_Recording\CR_CA1\LFP/LFp.pkl")
# Assuming df is the input DataFrame with 'animal_id' and 'lfp_py_norm_run' columns
# Step 1: Get unique animal IDs
unique_animals = np.unique(df['animal_id'])
print(f"Number of unique animals: {len(unique_animals)}")
print(f"Animal IDs: {unique_animals}")


fig = plt.figure(figsize=(7.2, 11), dpi=2400)
plt.rcParams.update({'font.size': 7,'font.family': 'DejaVu Sans'})
gs = gridspec.GridSpec(8, 8, height_ratios=[0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8], width_ratios=[0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8],wspace=3,hspace=3)  # First row taller

plt.rcParams.update({
    'axes.labelpad': -0.1,
    'ytick.major.pad': -0.1,
    'xtick.major.pad': -0.1,
    'ytick.major.size': 2,
    'xtick.major.size': 2
})

ax1 = fig.add_subplot(gs[0:2, 0:4])
ax2 = fig.add_subplot(gs[0:2, 4:8])
ax3 = fig.add_subplot(gs[2:4, 0:4])
ax4 = fig.add_subplot(gs[2:4, 4:8])


ax11 = fig.add_subplot(gs[4:6, 0:2])
ax12 = fig.add_subplot(gs[4:6, 2:4])
ax13 = fig.add_subplot(gs[4:6, 4:6])
ax14 = fig.add_subplot(gs[4:6, 6:8])

ax21 = fig.add_subplot(gs[6:8, 0:2])
ax22 = fig.add_subplot(gs[6:8, 2:4])
ax23 = fig.add_subplot(gs[6:8, 4:6])
ax24 = fig.add_subplot(gs[6:8, 6:8])


axes_event = [ax11, ax12 ,ax13 ,ax14, ax21,ax22 ,ax23 ,ax24]

# Define control and experimental animal IDs
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

# Define a common frequency grid (0 to 100 Hz, assuming 151 points for consistency)
common_frequencies = np.linspace(1, 151, 75)  # Adjust num_points if needed

# Step 2: Collect and average LFP data for each animal with index filtering
animal_lfp_averages = {}
frequency_indices = common_frequencies  # Use common frequencies for plotting

types = ["lfp_py_norm_run","lfp_sr_norm_run","lfp_py_norm_rest","lfp_sr_norm_rest"]
axes = [ax1,ax2,ax3,ax4]

for i,lfp in enumerate(types):
    ax = axes[i]
    for animal_id in unique_animals:
        # Filter DataFrame for the current animal
        animal_data = df[df['animal_id'] == animal_id][lfp]
        
        # Initialize a list to store filtered power values for this animal
        all_power_values = []
        
        # Iterate through each row's power vector
        for power_vector in animal_data:
            if isinstance(power_vector, list) and power_vector:
                # Assume power_vector[0] is a pandas Series with an index
                if isinstance(power_vector[0], pd.Series):
                    power_series = power_vector[0]
                    indices = power_series.index  # Use the Series index directly
                    power_values = power_series.values  # Get the power values
                    
                    # Reindex or interpolate to common_frequencies
                    if not np.array_equal(indices, common_frequencies):
                        # Interpolate to align with common_frequencies
                        interpolated_power = np.interp(
                            common_frequencies,
                            indices,
                            power_values,
                            left=np.nan,
                            right=np.nan
                        )
                    else:
                        interpolated_power = power_values
                    
                    # Filter for indices where 0 <= index <= 100 (already ensured by common_frequencies)
                    if len(interpolated_power) == len(common_frequencies):
                        all_power_values.append(interpolated_power)
                else:
                    print(f"Warning: power_vector[0] for animal {animal_id} is not a pandas Series, skipping.")
        
        # Compute the average power for this animal
        if all_power_values:  # Check if there are any values
            try:
                all_power_values = np.vstack(all_power_values)
                average_power = np.nanmean(all_power_values, axis=0)  # Average across trials, ignoring NaNs
            except ValueError as e:
                print(f"Error stacking arrays for animal {animal_id}: {e}")
                average_power = np.full(len(common_frequencies), np.nan)
        else:
            average_power = np.full(len(common_frequencies), np.nan)  # Handle cases with no data
        
        # Store the result
        animal_lfp_averages[animal_id] = average_power

    # Step 3: Create DataFrame with condition labels
    average_lfp_df = pd.DataFrame({
        "animal_id": animal_lfp_averages.keys(),
        "average_lfp_power": animal_lfp_averages.values()
    })

    # Add condition column
    average_lfp_df['condition'] = average_lfp_df['animal_id'].apply(
        lambda x: 'Control' if x in control_ids else 'Experimental' if x in exp_ids else 'Unknown'
    )

    # Filter out any animals not in control_ids or exp_ids
    average_lfp_df = average_lfp_df[average_lfp_df['condition'] != 'Unknown']

    # Prepare for statistical testing
    control_df = average_lfp_df[average_lfp_df['condition'] == 'Control']
    exp_df = average_lfp_df[average_lfp_df['condition'] == 'Experimental']

    control_powers = np.stack(control_df['average_lfp_power'].values)  # Shape: (num_control_animals, num_freqs)
    exp_powers = np.stack(exp_df['average_lfp_power'].values)  # Shape: (num_exp_animals, num_freqs)

    # Compute p-values using t-test for each frequency bin
    p_values = []
    num_freqs = control_powers.shape[1]
    for i in range(num_freqs):
        ctrl = control_powers[:, i]
        ex = exp_powers[:, i]
        ctrl = ctrl[~np.isnan(ctrl)]
        ex = ex[~np.isnan(ex)]
        if len(ctrl) >= 2 and len(ex) >= 2:
            _, p = scipy.stats.ttest_ind(ctrl, ex)
            p_values.append(p)
        else:
            p_values.append(np.nan)

    p_values = np.array(p_values)

    # FDR correction for multiple comparisons
    valid_mask = ~np.isnan(p_values)
    if np.any(valid_mask):
        reject, q_values, _, _ = multipletests(p_values[valid_mask], method='fdr_bh')
        q_full = np.full_like(p_values, np.nan)
        q_full[valid_mask] = q_values
    else:
        q_full = np.full_like(p_values, np.nan)

    # Identify significant frequencies (q < 0.05)
    significant_mask = q_full < 0.05
    significant_freqs = common_frequencies[significant_mask]

    # Step 4: Prepare data for plotting (convert to long format)
    plot_data = []
    for _, row in average_lfp_df.iterrows():
        animal_id = row['animal_id']
        condition = row['condition']
        power_vector = row['average_lfp_power']
        
        if isinstance(power_vector, np.ndarray) and not np.all(np.isnan(power_vector)):
            for freq_idx, power in zip(frequency_indices, power_vector):
                plot_data.append({
                    'animal_id': animal_id,
                    'condition': condition,
                    'frequency': freq_idx,  # Use common_frequencies for x-axis
                    'power': power
                })

    plot_df = pd.DataFrame(plot_data)

    # Step 5: Create figure with subplot (expand to more subplots as needed, e.g., fig, axs = plt.subplots(1, 2))
    sns.set(style="whitegrid")

    # Plot using lineplot on the specific ax
    sns.lineplot(
        data=plot_df,
        x='frequency',
        y='power',
        hue='condition',
        style='condition',
        palette={'Control': 'blue', 'Experimental': 'red'},
        markers=False,
        dashes=False,
        ax=ax
    )

    # Customize plot
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Normalized Power")
    ax.set_yscale('log')  # Set y-axis to logarithmic scale
    ax.set_ylim(1e-6, 1e-0)  # Set y-axis limits from 10^-6 to 10^0
    ax.set_xlim(0, 150)  # Adjust x-limit to 0-100 Hz as per original title
    ax.grid(False)
    ax.set_aspect('auto')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.legend().set_visible(False)
    # Add green dots at the top for significant differences
    if len(significant_freqs) > 0:
        y_pos = 5e-1  # Position near the top of the y-axis (adjust if needed based on data)
        ax.scatter(significant_freqs, [y_pos] * len(significant_freqs), color='green', s=20, marker='*')

    # Step 6: Display the DataFrame for reference
    print("\nAverage LFP DataFrame with Conditions:")
    print(average_lfp_df[['animal_id', 'condition']])

# Assuming df is the input DataFrame with 'animal_id' and 'lfp_py_norm_run' columns
# Step 1: Get unique animal IDs
unique_animals = np.unique(df['animal_id'])
print(f"Number of unique animals: {len(unique_animals)}")
print(f"Animal IDs: {unique_animals}")



import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro, ttest_ind, mannwhitneyu

# Assuming df is your DataFrame
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']


variables = ['slow_event_rate_py', 'slow_theta_gamma_coupling_py','fast_event_rate_py', 'fast_theta_gamma_coupling_py', 
            'slow_event_rate_sr', 'slow_theta_gamma_coupling_sr','fast_event_rate_sr', 'fast_theta_gamma_coupling_sr']

titles = ['Episods/s', 'Vector length','Episods/s', 'Vector length', 
            'Episods/s', 'Vector length', 'Episods/s', 'Vector length']


# Initialize an empty list to store results
data = []
unpacked_data=[]

for idx, row in df.iterrows():
    animal_id = row['animal_id']
    condition = "Control" if animal_id in control_ids else "Exp" if animal_id in exp_ids else None
    if condition:
        row_data = {"condition": condition, "row_id": idx}
        for var in variables:
            value = row[var]  # Assume scalar for simplicity
            row_data[var] = value
        data.append(row_data)

data_df = pd.DataFrame(data)
for _, row in data_df.iterrows():
    condition = row['condition']
    row_id = row['row_id']
    # Get the lists for each variable
    lists_per_var = {var: row[var] for var in variables}
    # Determine the length of the lists (assuming all lists in a row have the same length)
    list_length = len(lists_per_var[variables[0]]) if lists_per_var[variables[0]] else 0
    # Create a row for each index in the lists
    for i in range(list_length):
        new_row = {
            'condition': condition,
            'row_id': row_id,
            'list_index': i  # To track the position in the list
        }
        for var in variables:
            new_row[var] = lists_per_var[var][i] if i < len(lists_per_var[var]) else None
        unpacked_data.append(new_row)

# Create a new DataFrame from the unpacked data
unpacked_df = pd.DataFrame(unpacked_data)
control_color = 'blue'
exp_color = 'red'
# Create subplots

for i, var in enumerate(variables):
    # Create subplot
    ax = axes_event[i]
    # # Boxplot with specified colors
    # sns.boxplot(x='condition', y=var, data=unpacked_df, 
    #             palette={'Control': 'blue', 'Exp': 'cyan'})
    
    sns.violinplot(
            data=unpacked_df, x='condition', y=var, 
            ax=ax,
            inner = None,
            palette={"Control": control_color, "Exp": exp_color}, width=0.8, cut=0, linewidth=0
        )
    # Add individual points with matching colors
    sns.boxplot(
        data=unpacked_df, 
        x='condition', y=var, 
        palette={"Control": "black", "Exp": "black"},
        width=0.3, 
        fill=False,  # No fill, only outlines
        showfliers=False,  # Hide outliers
        showmeans=False,  # Remove mean marker, assuming midline is the median
        linewidth=1,  # Makes the lines narrower (thinner)
        ax=ax  # Add this
    )
    #ax.set_ylabel(titles[i])
    ax.set_xlabel('')
    ax.yaxis.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation = -45)
    y_max = ax.get_ylim()[1]

    ax.set_ylabel(titles[i])
    # Normality test
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    # Shapiro test for normality
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    # Choose statistical test based on normality (p < 0.05 indicates non-normal)
    if p_c > 0.05 and p_e > 0.05:
        # Both normal: use t-test
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        # At least one non-normal: use Mann-Whitney U
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    # Add title with statistical results
    y_max = ax.get_ylim()[1]
    bar_height = y_max * 0.1  # Adjust this value to position the bar above the plot
    x_positions = [0, 1]  # Adjusted positions for 'Control' and 'Experimental' groups
    p_val =  p_val
    if (p_val < 0.05) & (p_val > 0.01):
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'*', ha='center', va='bottom')
    elif (p_val < 0.01) & (p_val > 0.001):
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'**', ha='center', va='bottom')
    elif p_val < 0.001:
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'***', ha='center', va='bottom')


# Print detailed statistical results
print("\nStatistical Analysis Results:")
print("-" * 50)
for var in variables:
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    if p_c > 0.05 and p_e > 0.05:
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    print(f"\n{var}:")
    print(f"Control normality (Shapiro): p={p_c:.4f}")
    print(f"Exp normality (Shapiro): p={p_e:.4f}")
    print(f"{test_name}: statistic={stat:.4f}, p-value={p_val:.4f}")

    # Adjust layout and display
plt.tight_layout()
plt.savefig(r'Q:/sachuriga/CR_CA1_paper/Figures/suppfig10.png', transparent=True, dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:

# Assuming df is the input DataFrame with 'animal_id' and 'lfp_py_norm_run' columns
# Step 1: Get unique animal IDs
unique_animals = np.unique(df['animal_id'])
print(f"Number of unique animals: {len(unique_animals)}")
print(f"Animal IDs: {unique_animals}")

fig = plt.figure(figsize=(7.2, 11), dpi=2400)
plt.rcParams.update({'font.size': 7,'font.family': 'DejaVu Sans'})
gs = gridspec.GridSpec(8, 8, height_ratios=[0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8], width_ratios=[0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8],wspace=2,hspace=2)  # First row taller

plt.rcParams.update({
    'axes.labelpad': -0.1,
    'ytick.major.pad': -0.1,
    'xtick.major.pad': -0.1,
    'ytick.major.size': 2,
    'xtick.major.size': 2
})

ax1 = fig.add_subplot(gs[0:2, 0:4])
ax2 = fig.add_subplot(gs[0:2, 4:8])
ax3 = fig.add_subplot(gs[2:4, 0:4])
ax4 = fig.add_subplot(gs[2:4, 4:8])


ax11 = fig.add_subplot(gs[4:6, 0:2])
ax12 = fig.add_subplot(gs[4:6, 2:4])
ax13 = fig.add_subplot(gs[4:6, 4:6])
ax14 = fig.add_subplot(gs[4:6, 6:8])

ax21 = fig.add_subplot(gs[6:8, 0:2])
ax22 = fig.add_subplot(gs[6:8, 2:4])
ax23 = fig.add_subplot(gs[6:8, 4:6])
ax24 = fig.add_subplot(gs[6:8, 6:8])

axes_event = [ax11, ax12 ,ax13 ,ax14, ax21,ax22 ,ax23 ,ax24]

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro, ttest_ind, mannwhitneyu

# Assuming df is your DataFrame
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']


variables = ['slow_event_rate_py', 'slow_theta_gamma_coupling_py','fast_event_rate_py', 'fast_theta_gamma_coupling_py', 
            'slow_event_rate_sr', 'slow_theta_gamma_coupling_sr','fast_event_rate_sr', 'fast_theta_gamma_coupling_sr']

titles = ['Episods/s', 'Vector length','Episods/s', 'Vector length', 
            'Episods/s', 'Vector length', 'Episods/s', 'Vector length']


# Initialize an empty list to store results
data = []
unpacked_data=[]

for idx, row in df.iterrows():
    animal_id = row['animal_id']
    condition = "Control" if animal_id in control_ids else "Exp" if animal_id in exp_ids else None
    if condition:
        row_data = {"condition": condition, "row_id": idx}
        for var in variables:
            value = row[var]  # Assume scalar for simplicity
            row_data[var] = value
        data.append(row_data)

data_df = pd.DataFrame(data)
for _, row in data_df.iterrows():
    condition = row['condition']
    row_id = row['row_id']
    # Get the lists for each variable
    lists_per_var = {var: row[var] for var in variables}
    # Determine the length of the lists (assuming all lists in a row have the same length)
    list_length = len(lists_per_var[variables[0]]) if lists_per_var[variables[0]] else 0
    # Create a row for each index in the lists
    for i in range(list_length):
        new_row = {
            'condition': condition,
            'row_id': row_id,
            'list_index': i  # To track the position in the list
        }
        for var in variables:
            new_row[var] = lists_per_var[var][i] if i < len(lists_per_var[var]) else None
        unpacked_data.append(new_row)

# Create a new DataFrame from the unpacked data
unpacked_df = pd.DataFrame(unpacked_data)
control_color = 'blue'
exp_color = 'red'
# Create subplots

for i, var in enumerate(variables):
    # Create subplot
    ax = axes_event[i]
    # # Boxplot with specified colors
    # sns.boxplot(x='condition', y=var, data=unpacked_df, 
    #             palette={'Control': 'blue', 'Exp': 'cyan'})
    
    sns.violinplot(
            data=unpacked_df, x='condition', y=var, 
            ax=ax,
            inner = None,
            palette={"Control": control_color, "Exp": exp_color}, width=0.8, cut=0, linewidth=0
        )
    # Add individual points with matching colors
    sns.boxplot(
        data=unpacked_df, 
        x='condition', y=var, 
        palette={"Control": "black", "Exp": "black"},
        width=0.3, 
        fill=False,  # No fill, only outlines
        showfliers=False,  # Hide outliers
        showmeans=False,  # Remove mean marker, assuming midline is the median
        linewidth=1,  # Makes the lines narrower (thinner)
        ax=ax  # Add this
    )
    #ax.set_ylabel(titles[i])
    ax.set_xlabel('')
    ax.yaxis.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation = -45)
    y_max = ax.get_ylim()[1]

    ax.set_ylabel(titles[i])
    # Normality test
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    # Shapiro test for normality
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    # Choose statistical test based on normality (p < 0.05 indicates non-normal)
    if p_c > 0.05 and p_e > 0.05:
        # Both normal: use t-test
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        # At least one non-normal: use Mann-Whitney U
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    # Add title with statistical results
    y_max = ax.get_ylim()[1]
    bar_height = y_max * 0.1  # Adjust this value to position the bar above the plot
    x_positions = [0, 1]  # Adjusted positions for 'Control' and 'Experimental' groups
    p_val =  p_val
    if (p_val < 0.05) & (p_val > 0.01):
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'*', ha='center', va='bottom')
    elif (p_val < 0.01) & (p_val > 0.001):
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'**', ha='center', va='bottom')
    elif p_val < 0.001:
        ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                            color='black', lw=1.5)
        ax.text(0.5, y_max + bar_height * 1.1, f'***', ha='center', va='bottom')


# Print detailed statistical results
print("\nStatistical Analysis Results:")
print("-" * 50)
for var in variables:
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    if p_c > 0.05 and p_e > 0.05:
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    print(f"\n{var}:")
    print(f"Control normality (Shapiro): p={p_c:.4f}")
    print(f"Exp normality (Shapiro): p={p_e:.4f}")
    print(f"{test_name}: statistic={stat:.4f}, p-value={p_val:.4f}")

    # Adjust layout and display
plt.tight_layout()
plt.savefig(r'Q:/sachuriga/CR_CA1_paper/Figures/suppfig10.png', transparent=True, dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro, ttest_ind, mannwhitneyu

# Assuming df is your DataFrame
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']


variables = ['slow_event_rate_py', 'slow_theta_gamma_coupling_py','fast_event_rate_py', 'fast_theta_gamma_coupling_py', 
            'slow_event_rate_sr', 'slow_theta_gamma_coupling_sr','fast_event_rate_sr', 'fast_theta_gamma_coupling_sr']



# Initialize an empty list to store results
data = []
unpacked_data=[]

for idx, row in df.iterrows():
    animal_id = row['animal_id']
    condition = "Control" if animal_id in control_ids else "Exp" if animal_id in exp_ids else None
    if condition:
        row_data = {"condition": condition, "row_id": idx}
        for var in variables:
            value = row[var]  # Assume scalar for simplicity
            row_data[var] = value
        data.append(row_data)

data_df = pd.DataFrame(data)
for _, row in data_df.iterrows():
    condition = row['condition']
    row_id = row['row_id']
    # Get the lists for each variable
    lists_per_var = {var: row[var] for var in variables}
    # Determine the length of the lists (assuming all lists in a row have the same length)
    list_length = len(lists_per_var[variables[0]]) if lists_per_var[variables[0]] else 0
    # Create a row for each index in the lists
    for i in range(list_length):
        new_row = {
            'condition': condition,
            'row_id': row_id,
            'list_index': i  # To track the position in the list
        }
        for var in variables:
            new_row[var] = lists_per_var[var][i] if i < len(lists_per_var[var]) else None
        unpacked_data.append(new_row)

# Create a new DataFrame from the unpacked data
unpacked_df = pd.DataFrame(unpacked_data)
control_color = 'blue'
exp_color = 'red'
# Create subplots

for i, var in enumerate(variables, 1):
    # Create subplot
    ax = axes_event[i]
    # # Boxplot with specified colors
    # sns.boxplot(x='condition', y=var, data=unpacked_df, 
    #             palette={'Control': 'blue', 'Exp': 'cyan'})
    
    sns.violinplot(
            data=unpacked_df, x='condition', y=var, 
            #ax=ax,
            inner = None,
            palette={"Control": control_color, "Exp": exp_color}, width=0.8, cut=0, linewidth=0
        )
    # Add individual points with matching colors
    sns.boxplot(
        data=unpacked_df, 
        x='condition', y=var, 
        palette={"Control": "black", "Exp": "black"},
        width=0.3, 
        fill=False,  # No fill, only outlines
        showfliers=False,  # Hide outliers
        showmeans=False,  # Remove mean marker, assuming midline is the median
        linewidth=1,  # Makes the lines narrower (thinner)
        ax=ax  # Add this
    )
    #ax.set_ylabel(titles[idx])

    ax.set_xlabel('')
    ax.yaxis.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation = -45)
    y_max = ax.get_ylim()[1]


    # Normality test
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    # Shapiro test for normality
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    # Choose statistical test based on normality (p < 0.05 indicates non-normal)
    if p_c > 0.05 and p_e > 0.05:
        # Both normal: use t-test
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        # At least one non-normal: use Mann-Whitney U
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    # Add title with statistical results
    plt.title(f'{var}\n{test_name}: p={p_val:.4f}', fontsize=10)
    
    # Adjust y-label
    plt.ylabel(var.split('_')[0] + '_' + var.split('_')[1])

# Adjust layout and display
plt.tight_layout()
plt.show()

# Print detailed statistical results
print("\nStatistical Analysis Results:")
print("-" * 50)
for var in variables:
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    if p_c > 0.05 and p_e > 0.05:
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    print(f"\n{var}:")
    print(f"Control normality (Shapiro): p={p_c:.4f}")
    print(f"Exp normality (Shapiro): p={p_e:.4f}")
    print(f"{test_name}: statistic={stat:.4f}, p-value={p_val:.4f}")